# 14 为什么优化器要使用参数组学习率？

## 面试回答主线

参数组学习率允许 embedding、主干矩阵、输出头、LoRA 或 norm/bias 使用不同步长与正则策略。合理分组来自参数角色和更新尺度，而不是“层号越大学习率越高”的固定规则。面试中要说明 group 覆盖完整且互不重复、断点续训中 group 顺序稳定，以及每组都要记录更新 RMS。本实验手写一个工单分类头的 embedding、主干和输出参数，比较全参数同 LR 与分组 LR 的相对更新，并演示误把 norm scale 放进高 LR 组的漂移。

**核心公式：** 第 $k$ 组参数更新为 $\theta_k\leftarrow\theta_k-\eta_k u_k$。真正要比较的是 $\lVert\Delta\theta_k\rVert/(\lVert\theta_k\rVert+\epsilon)$，而非只看绝对学习率。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
embed = torch.tensor([0.40, -0.20, 0.30], requires_grad=True)  # 创建模拟 token embedding 参数。
backbone = torch.tensor([[0.20, -0.10], [0.15, 0.05], [-0.20, 0.12]], requires_grad=True)  # 创建模拟主干矩阵。
head = torch.tensor([[0.10, -0.08], [0.06, 0.11]], requires_grad=True)  # 创建模拟输出头。
norm_scale = torch.tensor([1.0, 1.0], requires_grad=True)  # 创建模拟 RMSNorm 缩放参数。
logits = (features + embed) @ backbone @ head * norm_scale  # 手写带 embedding、主干、输出和 norm 的 logits。
loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算同一工单分类目标。
all_grads = torch.autograd.grad(loss, [embed, backbone, head, norm_scale])  # 取得所有参数梯度。
same_lr_updates = [0.30 * gradient for gradient in all_grads]  # 让所有参数使用同一学习率作为基线。
baseline_metric = float(same_lr_updates[3].norm() / norm_scale.norm())  # 记录 norm 参数相对更新幅度。
print(f'同 LR：loss={loss.item():.4f}，norm 相对更新={baseline_metric:.5f}，四组更新范数={ [round(float(value.norm()), 4) for value in same_lr_updates] }')  # 展示基线更新失衡。


同 LR：loss=0.7050，norm 相对更新=0.00200，四组更新范数=[0.0001, 0.0158, 0.0291, 0.0028]


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
group_lrs = {'embedding': 0.06, 'backbone': 0.20, 'head': 0.28, 'norm': 0.02}  # 明确声明每组学习率。
named_grads = {'embedding': all_grads[0], 'backbone': all_grads[1], 'head': all_grads[2], 'norm': all_grads[3]}  # 将梯度与参数角色绑定。
group_updates = {name: group_lrs[name] * gradient for name, gradient in named_grads.items()}  # 手写每组不同尺度的更新。
relative_updates = {'embedding': float(group_updates['embedding'].norm() / embed.norm()), 'backbone': float(group_updates['backbone'].norm() / backbone.norm()), 'head': float(group_updates['head'].norm() / head.norm()), 'norm': float(group_updates['norm'].norm() / norm_scale.norm())}  # 计算各组相对更新。
core_metric = relative_updates['norm']  # 保存 norm 组受保护后的相对更新。
print(f'参数组 LR={group_lrs}')  # 输出可审计的分组配置。
print(f'相对更新={ {name: round(value, 5) for name, value in relative_updates.items()} }')  # 展示分组后各角色更新尺度。


参数组 LR={'embedding': 0.06, 'backbone': 0.2, 'head': 0.28, 'norm': 0.02}
相对更新={'embedding': 2e-05, 'backbone': 0.02919, 'head': 0.15177, 'norm': 0.00013}


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.001998
核心机制     | 指标=0.000133


## 结果解读

基线和核心输出只在本受控案例中比较。生产中需要在日志里输出每组参数数、梯度范数、更新 RMS 和重叠检查；新增 LoRA/adapter 时必须显式决定属于哪组。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
bad_norm_update = 1.50 * all_grads[3]  # 故意把 norm scale 放进异常高学习率组。
failure_metric = float(bad_norm_update.norm() / norm_scale.norm())  # 记录错误分组导致的相对漂移。
fix_metric = relative_updates['norm']  # 复用低学习率 norm 组的相对漂移。
print(f'失败：norm 误入高 LR 组，相对更新={failure_metric:.5f}；修复：专属 norm 组={fix_metric:.5f}')  # 展示参数角色分组价值。


失败：norm 误入高 LR 组，相对更新=0.00999；修复：专属 norm 组=0.00013


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产中需要在日志里输出每组参数数、梯度范数、更新 RMS 和重叠检查；新增 LoRA/adapter 时必须显式决定属于哪组。

**常见坑：** 参数遗漏会静默不训练，参数重叠会被更新两次；只按名称匹配也容易把 norm/bias 放错组。

**延伸追问：** 全参微调、LoRA 和 continued pretraining 的 group 配方为何不同？如何验证 checkpoint 恢复后 group 顺序没有变化？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert set(group_lrs) == {'embedding', 'backbone', 'head', 'norm'}  # 验证参数组覆盖四类参数。
assert core_metric < baseline_metric  # 验证专属 norm 学习率减小其相对更新。
assert failure_metric > fix_metric  # 验证错误高 LR 组会放大 norm 漂移。
assert all(value > 0.0 for value in group_lrs.values())  # 验证所有训练参数组具有正学习率。
